# 🚗 Car Price Prediction — Milestone 2: Data Preparation

**Dataset:** Car Price Prediction Challenge (Kaggle)  
**Task:** Supervised Regression — Predicting used car market value  
**Notebook structure:**
1. Load & inspect raw data
2. Data cleaning & missing value handling
3. Train/test split (stratified)
4. Feature engineering (hand-crafted + encoded)
5. Outlier analysis & filtering
6. Scaling & pre-processing
7. Feature selection
8. Summary

## 0. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_regression, SelectKBest

# ── Reproducibility ──────────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Plot style ───────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

print('All imports successful. Random seed set to', RANDOM_SEED)

## 1. Load & Inspect Raw Data

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────────
# Download from Kaggle: https://www.kaggle.com/datasets/deepcontractor/car-price-prediction-challenge
df_raw = pd.read_csv('car_price_dataset.csv')   # <-- adjust path if needed

print('Shape:', df_raw.shape)
df_raw.head()

In [ ]:
# ── Basic info ────────────────────────────────────────────────────────────────
df_raw.info()

In [ ]:
# ── Missing / hidden-null summary ─────────────────────────────────────────────
# The dataset uses '-' as a null placeholder in some columns (e.g. Levy)
missing_real = df_raw.isnull().sum()
missing_dash = (df_raw == '-').sum()

miss_df = pd.DataFrame({
    'NaN count':   missing_real,
    'Dash count':  missing_dash,
    'Total missing': missing_real + missing_dash
})
miss_df['Missing %'] = (miss_df['Total missing'] / len(df_raw) * 100).round(2)
miss_df = miss_df[miss_df['Total missing'] > 0].sort_values('Missing %', ascending=False)
print(miss_df)

# ── Visualize missing values ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(miss_df.index, miss_df['Missing %'], color=sns.color_palette('muted')[0])
ax.set_xlabel('Missing Values (%)')
ax.set_title('Percentage of Missing / Placeholder Values per Feature')
for i, v in enumerate(miss_df['Missing %']):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center')
plt.tight_layout()
plt.savefig('fig_missing_values.png', bbox_inches='tight')
plt.show()
print('Figure saved: fig_missing_values.png')

**Observation:** The `Levy` column has the highest share of missing values (~30%), represented as dashes rather than `NaN`. This requires special handling before any numeric conversion.

## 2. Data Cleaning

We work on a **copy** of the raw data to preserve the original for reference.

In [ ]:
df = df_raw.copy()

# ── 2.1 Replace dash placeholders with NaN ────────────────────────────────────
df.replace('-', np.nan, inplace=True)

# ── 2.2 Clean Levy (tax): remove commas, cast to float ────────────────────────
df['Levy'] = df['Levy'].astype(str).str.replace(',', '', regex=False)
df['Levy'] = pd.to_numeric(df['Levy'], errors='coerce')

# ── 2.3 Clean Mileage: strip ' km', cast to int ───────────────────────────────
df['Mileage'] = df['Mileage'].astype(str).str.replace(' km', '', regex=False).str.replace(',', '', regex=False)
df['Mileage'] = pd.to_numeric(df['Mileage'], errors='coerce')

# ── 2.4 Clean Engine volume: extract numeric part (handles 'Turbo') ────────────
df['Engine volume'] = df['Engine volume'].astype(str).str.extract(r'([0-9]*\.?[0-9]+)')[0]
df['Engine volume'] = pd.to_numeric(df['Engine volume'], errors='coerce')

# ── 2.5 Clean Price: remove commas, cast to float ─────────────────────────────
df['Price'] = df['Price'].astype(str).str.replace(',', '', regex=False)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# ── 2.6 Drop rows where Price is NaN (target cannot be missing) ───────────────
before = len(df)
df.dropna(subset=['Price'], inplace=True)
print(f'Rows dropped due to missing Price: {before - len(df)}')

# ── 2.7 Impute Levy with median ───────────────────────────────────────────────
# Median is used because Levy is right-skewed; median is more robust than mean.
levy_median = df['Levy'].median()
df['Levy'].fillna(levy_median, inplace=True)
print(f'Levy missing values imputed with median: {levy_median:.0f}')

# ── 2.8 Impute remaining numeric NaNs with median ─────────────────────────────
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    if df[col].isnull().sum() > 0:
        med = df[col].median()
        df[col].fillna(med, inplace=True)
        print(f'  {col}: {df[col].isnull().sum()} NaNs filled with median={med:.2f}')

# ── 2.9 Drop rows where any categorical column is NaN ─────────────────────────
cat_cols = df.select_dtypes(include='object').columns.tolist()
before = len(df)
df.dropna(subset=cat_cols, inplace=True)
print(f'\nRows dropped due to missing categorical values: {before - len(df)}')
print(f'Clean dataset shape: {df.shape}')

In [ ]:
# ── 2.10 Remove obviously erroneous prices (e.g. $1 listings) ─────────────────
# A price of $1 is almost certainly a data entry error.
# We apply a LOWER bound of $200 only — we do NOT remove high-priced luxury cars
# because those are plausible real-world data points.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Price'], bins=80, color=sns.color_palette('muted')[1], edgecolor='white')
axes[0].set_title('Price Distribution — Before Filtering')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

low_threshold = 200
n_before = len(df)
df = df[df['Price'] >= low_threshold]
n_after = len(df)
print(f'Removed {n_before - n_after} rows with Price < ${low_threshold}')

axes[1].hist(df['Price'], bins=80, color=sns.color_palette('muted')[2], edgecolor='white')
axes[1].set_title('Price Distribution — After Removing $<200 Entries')
axes[1].set_xlabel('Price ($)')
axes[1].set_ylabel('Count')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Effect of Lower-Bound Outlier Removal on Price', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_price_outlier_filter.png', bbox_inches='tight')
plt.show()

**Note on high-price outliers:** Unusually expensive vehicles (luxury/exotic cars) are **not removed**. Unlike \$1 listings which are clearly erroneous, a \$100,000+ car is a legitimate real-world listing. Removing them would bias the model to underpredict for premium vehicles.

## 3. Train / Test Split

We split **before** any feature engineering to prevent data leakage. The test set is never used to fit transformers.

Because the `Price` distribution is right-skewed, we apply a **log-binning strategy** so that rare high-price samples appear in both splits proportionally.

In [ ]:
# ── Create price bins for stratification ──────────────────────────────────────
df['price_bin'] = pd.qcut(df['Price'], q=10, labels=False, duplicates='drop')

X = df.drop(columns=['Price', 'price_bin'])
y = df['Price']
strat = df['price_bin']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=strat
)

print(f'Training set:  {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test set:      {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)')

# ── Verify distribution similarity ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(np.log1p(y_train), bins=40, alpha=0.6, label='Train', color=sns.color_palette('muted')[0])
ax.hist(np.log1p(y_test),  bins=40, alpha=0.6, label='Test',  color=sns.color_palette('muted')[3])
ax.set_xlabel('log(1 + Price)')
ax.set_ylabel('Count')
ax.set_title('Log-Price Distribution: Train vs Test (stratified split)')
ax.legend()
plt.tight_layout()
plt.savefig('fig_train_test_distribution.png', bbox_inches='tight')
plt.show()

**Why 80/20?** With ~19,000 samples this gives us ~15,200 training and ~3,800 test rows — enough for robust model evaluation while retaining sufficient training data.

## 4. Feature Engineering

All transformations are derived **from training data only**, then applied to both sets.

In [ ]:
# ── Helper: apply same transforms to train and test ───────────────────────────
def engineer_features(df_in):
    df_e = df_in.copy()

    # ── Hand-crafted feature 1: Vehicle Age ───────────────────────────────────
    # Intuition: Newer cars are worth more. Age captures depreciation better
    # than raw production year.
    CURRENT_YEAR = 2024
    df_e['Vehicle_Age'] = CURRENT_YEAR - df_e['Prod. year'].astype(int)
    df_e['Vehicle_Age'] = df_e['Vehicle_Age'].clip(lower=0)  # no negative ages

    # ── Hand-crafted feature 2: Mileage per Year ──────────────────────────────
    # Intuition: A car with 200,000 km driven over 20 years is better maintained
    # than the same mileage in 5 years. Normalises wear rate.
    df_e['Mileage_Per_Year'] = df_e['Mileage'] / (df_e['Vehicle_Age'] + 1)  # +1 to avoid div/0

    # ── Hand-crafted feature 3: Engine Power Ratio ────────────────────────────
    # Intuition: A larger engine in a lighter/older car signals a sports/performance
    # vehicle. Ratio of engine volume to vehicle age acts as a 'performance proxy'.
    df_e['Engine_Age_Ratio'] = df_e['Engine volume'] / (df_e['Vehicle_Age'] + 1)

    # ── Hand-crafted feature 4 (bonus): Is Turbo ─────────────────────────────
    # Turbo engines tend to command a price premium. Extracted from 'Engine volume'
    # raw string before unit-stripping was applied.
    if 'Engine volume' in df_in.columns:
        raw_engine = df_raw.loc[df_in.index, 'Engine volume'].astype(str)
        df_e['Is_Turbo'] = raw_engine.str.lower().str.contains('turbo').astype(int)

    return df_e

X_train_fe = engineer_features(X_train)
X_test_fe  = engineer_features(X_test)

new_features = ['Vehicle_Age', 'Mileage_Per_Year', 'Engine_Age_Ratio', 'Is_Turbo']
print('New features added:', new_features)
X_train_fe[new_features].describe()

### 4.1 Analysis of New Hand-Crafted Features

In [ ]:
# ── Distribution plots for new features ───────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
train_with_price = X_train_fe.copy()
train_with_price['Price'] = y_train.values

for ax, feat in zip(axes.flatten(), new_features):
    ax.hist(X_train_fe[feat].dropna(), bins=40,
            color=sns.color_palette('muted')[new_features.index(feat)],
            edgecolor='white')
    ax.set_title(f'Distribution of {feat}')
    ax.set_xlabel(feat)
    ax.set_ylabel('Count')

plt.suptitle('Distribution of Hand-Crafted Features (Training Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_new_feature_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Scatter plots: new features vs Price ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

scatter_feats = ['Vehicle_Age', 'Mileage_Per_Year', 'Engine_Age_Ratio']
for ax, feat in zip(axes, scatter_feats):
    ax.scatter(X_train_fe[feat], y_train, alpha=0.15, s=10,
               color=sns.color_palette('muted')[scatter_feats.index(feat)])
    ax.set_xlabel(feat)
    ax.set_ylabel('Price ($)')
    ax.set_title(f'{feat} vs Price')
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('New Features vs Target Price (Training Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_new_features_vs_price.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Box plot: Is_Turbo vs Price ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
turbo_df = pd.DataFrame({'Is_Turbo': X_train_fe['Is_Turbo'].map({0: 'Non-Turbo', 1: 'Turbo'}),
                          'Price': y_train.values})
sns.boxplot(data=turbo_df, x='Is_Turbo', y='Price', palette='muted', ax=ax,
            showfliers=False)  # hide extreme outliers for readability
ax.set_title('Price Distribution: Turbo vs Non-Turbo Engines')
ax.set_xlabel('Engine Type')
ax.set_ylabel('Price ($)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('fig_turbo_vs_price.png', bbox_inches='tight')
plt.show()

turbo_df.groupby('Is_Turbo')['Price'].describe().round(0)

## 5. Encoding Categorical Variables

Scikit-learn requires numeric input. We use **Label Encoding** for ordinal-like high-cardinality features and **One-Hot Encoding** for low-cardinality nominal features.

In [ ]:
# ── Identify categorical columns ──────────────────────────────────────────────
cat_cols_fe = X_train_fe.select_dtypes(include='object').columns.tolist()
print('Categorical columns to encode:', cat_cols_fe)

# Separate: low-cardinality → OHE, high-cardinality → Label Encode
cardinality = {col: X_train_fe[col].nunique() for col in cat_cols_fe}
print('\nCardinality per column:')
for k, v in sorted(cardinality.items(), key=lambda x: x[1]):
    print(f'  {k}: {v} unique values')

In [ ]:
# ── Label encode high-cardinality features (Manufacturer, Model, Color) ──────
high_card = [col for col, card in cardinality.items() if card > 10]
low_card  = [col for col, card in cardinality.items() if card <= 10]

print('Label Encoding (high cardinality):', high_card)
print('One-Hot Encoding (low cardinality):', low_card)

le_dict = {}
for col in high_card:
    le = LabelEncoder()
    X_train_fe[col] = le.fit_transform(X_train_fe[col].astype(str))
    # Handle unseen values in test set
    X_test_fe[col] = X_test_fe[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    le_dict[col] = le

# ── One-hot encode low-cardinality features ───────────────────────────────────
X_train_fe = pd.get_dummies(X_train_fe, columns=low_card, drop_first=True)
X_test_fe  = pd.get_dummies(X_test_fe,  columns=low_card, drop_first=True)

# Align columns (test may miss some dummies if rare category not present)
X_train_fe, X_test_fe = X_train_fe.align(X_test_fe, join='left', axis=1, fill_value=0)

print(f'\nFeature count after encoding: {X_train_fe.shape[1]}')

## 6. Scaling

We use **RobustScaler** (scales using the interquartile range) rather than StandardScaler because car price data still contains significant skew. RobustScaler is less sensitive to extreme values.

In [ ]:
# ── Scale numeric features ────────────────────────────────────────────────────
# Fit ONLY on training data to avoid leakage
numeric_cols = X_train_fe.select_dtypes(include=[np.number]).columns.tolist()

scaler = RobustScaler()
X_train_scaled = X_train_fe.copy()
X_test_scaled  = X_test_fe.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train_fe[numeric_cols])
X_test_scaled[numeric_cols]  = scaler.transform(X_test_fe[numeric_cols])

# ── Visualize before/after scaling for key features ──────────────────────────
key_feats = ['Mileage', 'Vehicle_Age', 'Levy']
fig, axes = plt.subplots(2, 3, figsize=(13, 7))

for i, feat in enumerate(key_feats):
    axes[0, i].hist(X_train_fe[feat], bins=40,
                    color=sns.color_palette('muted')[i], edgecolor='white')
    axes[0, i].set_title(f'{feat} — Before Scaling')
    axes[0, i].set_xlabel(feat)
    axes[0, i].set_ylabel('Count')

    axes[1, i].hist(X_train_scaled[feat], bins=40,
                    color=sns.color_palette('muted')[i+3], edgecolor='white')
    axes[1, i].set_title(f'{feat} — After RobustScaler')
    axes[1, i].set_xlabel(f'{feat} (scaled)')
    axes[1, i].set_ylabel('Count')

plt.suptitle('Effect of RobustScaler on Key Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_scaling_comparison.png', bbox_inches='tight')
plt.show()

## 7. Feature Selection

We use **Mutual Information** regression to rank features by their statistical dependence with the target variable. This is model-agnostic and handles non-linear relationships.

In [ ]:
# ── Compute mutual information ────────────────────────────────────────────────
mi_scores = mutual_info_regression(
    X_train_scaled.fillna(0),
    y_train,
    random_state=RANDOM_SEED
)

mi_df = pd.Series(mi_scores, index=X_train_scaled.columns)\
          .sort_values(ascending=False)

# ── Plot top 20 features ──────────────────────────────────────────────────────
top_n = 20
fig, ax = plt.subplots(figsize=(9, 7))
mi_df.head(top_n).sort_values().plot.barh(ax=ax, color=sns.color_palette('muted')[0])
ax.set_title(f'Top {top_n} Features by Mutual Information Score')
ax.set_xlabel('Mutual Information Score')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig('fig_mutual_information.png', bbox_inches='tight')
plt.show()

print('Top 10 features:', mi_df.head(10).index.tolist())

In [ ]:
# ── Select top K features ─────────────────────────────────────────────────────
# We keep features with MI > 10% of max MI score (data-driven threshold)
threshold = mi_df.max() * 0.10
selected_features = mi_df[mi_df >= threshold].index.tolist()
print(f'Features selected (MI >= {threshold:.4f}): {len(selected_features)}')
print(selected_features)

X_train_final = X_train_scaled[selected_features]
X_test_final  = X_test_scaled[selected_features]

print(f'\nFinal training set shape: {X_train_final.shape}')
print(f'Final test set shape:     {X_test_final.shape}')

## 8. Correlation Heatmap — Final Feature Set

In [ ]:
# ── Pearson correlation on final features + target ────────────────────────────
final_with_price = X_train_final.copy()
final_with_price['Price'] = y_train.values

corr = final_with_price.corr()

fig, ax = plt.subplots(figsize=(max(10, len(selected_features)), max(8, len(selected_features)-2)))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Correlation Matrix — Final Feature Set + Price', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_correlation_heatmap_final.png', bbox_inches='tight')
plt.show()

## 9. Save Processed Data

In [ ]:
# ── Export for use in Milestone 3 ─────────────────────────────────────────────
X_train_final.to_csv('X_train_final.csv', index=False)
X_test_final.to_csv('X_test_final.csv',   index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv',   index=False)

print('Saved:')
print('  X_train_final.csv —', X_train_final.shape)
print('  X_test_final.csv  —', X_test_final.shape)
print('  y_train.csv       —', y_train.shape)
print('  y_test.csv        —', y_test.shape)

## 10. Summary

| Step | Action | Result |
|------|--------|--------|
| Data Cleaning | Replaced `-` with NaN, stripped units, cast types | Clean numeric columns |
| Missing Values | Median imputation for `Levy`; drop rows for categorical NaN | No missing values remain |
| Outlier Filtering | Removed prices < $200 (erroneous entries only) | Preserved high-price luxury cars |
| Train/Test Split | 80/20, stratified by price quantile bin | Balanced target distribution |
| Feature Engineering | 4 new features: `Vehicle_Age`, `Mileage_Per_Year`, `Engine_Age_Ratio`, `Is_Turbo` | Added domain-relevant predictors |
| Encoding | Label encoding (high-cardinality) + OHE (low-cardinality) | Fully numeric feature matrix |
| Scaling | RobustScaler (fit on train only) | Mitigates skew, prevents leakage |
| Feature Selection | Mutual Information threshold (10% of max) | Reduced to most informative features |

The processed dataset is ready for model training in **Milestone 3**.